# Agentic X — Integration & Export — runnable MOCK adapter

**CAI461 | Studio 67 | Parallel team development**

This notebook includes **runnable mock-code cells**. It is not a completed AI or MAS implementation.

## Purpose & limitations

> **MOCK DEVELOPMENT MODE — NOT REAL AI.** These cells can run before Notebook 04 exists. All results are synthetic and must be replaced with actual engine results before evaluation and submission.

**Dependencies:** the shared `core/mock_engine.py` module committed in this GitHub repository. No API key or LangSmith is required for these mock cells.

**Team rule:** Work on your own branch; don't treat MOCK output as actual model performance.

**Notebook 07 responsibility:** agree on the UI-facing event contract and validate it using synthetic output until Notebook 04 is available. This is **not the final Streamlit UI**.

## Connect to shared engine

The adapter uses the same public `run_scenario` function that Notebook 05 uses.

In [ ]:
# Clone this repository into the fresh Colab runtime if needed.
import os, sys, subprocess
repo_dir = "/content/Agentic-X" if os.path.isdir("/content") else os.path.abspath("Agentic-X")
if not os.path.isfile(os.path.join(repo_dir, "core", "mock_engine.py")):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/RENADBHAI/Agentic-X.git", repo_dir],
        check=True,
    )
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
from core.mock_engine import MockEngine, MOCK_CASES, get_mock_case, run_scenario
engine = MockEngine()
print("MOCK engine ready —", len(MOCK_CASES), "demo scenarios only")

## Define an interface adapter

The adapter is deliberately engine-agnostic. The real engine should return the same RunResult fields.

In [ ]:
def ui_payload(case_id: str, engine, overrides=None) -> dict:
    result = run_scenario(case_id, engine, overrides=overrides)
    return {
        "run_id": result["run_id"],
        "case_id": result["case_id"],
        "status": result["status"],
        "selected_option_id": result["selected_option_id"],
        "reason": result["reason"],
        "agent_status": {
            agent: "MOCK_COMPLETE" if result["is_mock"] else "COMPLETE"
            for agent in ("creative", "finance", "operations", "safety")
        },
        "messages": result["messages"],
        "events": result["events"],
        "is_mock": result["is_mock"],
    }

## Inspect Streamlit-ready payload

Test that the mock result can drive UI status, bubbles and a negotiation log while keeping a visible MOCK label.

In [ ]:
from pprint import pprint
demo_payload = ui_payload("SCR-DEMO", engine)
assert demo_payload["is_mock"] is True
assert len(demo_payload["agent_status"]) == 4
assert demo_payload["messages"]
print("MOCK UI adapter ready:")
pprint({k: v for k, v in demo_payload.items() if k not in {"messages", "events"}})
print("First event:", demo_payload["events"][0])

## User controls and failure states

Validate the same budget/time overrides expected from controls, including human escalation.

In [ ]:
ui_escalation = ui_payload("SCR-DEMO", engine, {"budget_sar": 0, "time_left_min": 0})
assert ui_escalation["status"] == "ESCALATED"
assert ui_escalation["selected_option_id"] is None
print("MOCK control override correctly triggered escalation.")

## Real integration checklist

Replace `MockEngine()` with the real engine, connect an actual Streamlit UI with running controls, show truthful ACL events and live status, test real API / LangSmith errors, and capture screenshots of three **real** scenarios.